In [1]:
%pip install sympy matplotlib scipy astropy sep opencv-python pyproj -q

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


First define all necessary functions from SO(3)

In [2]:
# SO(3) functions 
import numpy as np
def so3_wedge(w):
    w = w.flatten()
    return np.array([
        [0, -w[2], w[1]],
        [w[2], 0, -w[0]],
        [-w[1], w[0], 0]
    ])

def so3_exp(w):
    w = w.flatten()
    theta = np.linalg.norm(w)
    I = np.eye(3)
    
    if theta < 1e-7:
        return I
    
    w_hat = w / theta
    w_wedge = so3_wedge(w_hat)
    
    # Rodrigues' Rotation Formula
    R = I + np.sin(theta) * w_wedge + (1 - np.cos(theta)) * (w_wedge @ w_wedge)
    return R

def so3_vee(Phi):
    # Extracts [w1, w2, w3] from the skew-symmetric form
    return np.array([Phi[2, 1], Phi[0, 2], Phi[1, 0]])

def ad_so3(R):
    # For SO(3), Adjoint(R) is simply R
    return R
    
def so3_log(R):
    I = np.eye(3)
    cos_theta = np.clip((np.trace(R) - 1) / 2, -1.0, 1.0)
    theta = np.arccos(cos_theta)
    
    if abs(theta) < 1e-7:
        return np.zeros(3)
    
    # Extract the skew-symmetric part
    w_wedge = (theta / (2 * np.sin(theta))) * (R - R.T)
    return so3_vee(w_wedge)

Define the Equivariant Filter class

In [ ]:
import numpy as np

class Equivariant:

    def __init__(self, dt, sigma_x, sigma_y, sigma_z, sigma_gravity, Xinit):
        self.dt = dt

        # state (world frame)
        self.xi = np.array([0,0,1]) # state in world frame ?
        self.X = so3_exp(self.xi)

        self.xi0 = np.array([0,0,1]) 
        self.X0 = so3_exp(self.xi0)
    
        # initial condition for estimate
        self.ghat = np.array([0,0,1])
        # process noise covariance
        self.Q = np.eye(3) * sigma_gravity
        # sensor noise covariance
        self.Rcov = np.diag([sigma_x, sigma_y, sigma_z])
        self.B = np.array([[1, 0, 0], 
              [0, 1, 0]])
        self.Xhat = Xinit
        self.Delta = np.zeros(3)
        self.Rlocal = self.B @ self.Rcov @ self.B.T

        # initialize ricatti term
        self.sigma = np.eye(2) *0.1 # represents the initial uncertainty (of calculated initial states)
        self.Cts = np.array([[0,1],[-1,0]])

    # functions for easier computation: 
    def lift(self,in1, in2): # lift S2 element to lie algebra (go from g to xi)
        return so3_exp(in2)  
    def unlift(self,A): # lie algebra to S2 (go from xi to g)
        return so3_log(A)
    
    def phi(self, in1, in2): # left group action
        return in1.T @ in2
    def psi(self, in1, in2): # right group action
        return in1.T @ in2
    
    def left_mult(self, X, A): # left multiplication
        return A@X
    def right_mult(self,X, B): # right multiplication
        return X@B
    
    def right_inverse(self,A):
        return A.T @ np.linalg.inv(A@A.T)
    
    def theta(self,e): # local coordinate chart for S2
        e1 = np.array([0,0,1]) # origin   
        cross = np.cross(e1, e)
        dot = np.dot(e1, e)
        arr = np.array([[0,1,0],[0,0,1]])
        den = np.linalg.norm(cross)
        if den < 1e-12:
            return np.zeros(2)
        
        axis = cross / den
        angle = np.atan2(den, dot)
        t = angle * np.array([axis[0], axis[1]])
        return t
    
    def theta_inv(self, epsilon):
        e1 = np.array([0,0,1]).T
        vec = np.array([epsilon[0], epsilon[1], 0])
        tinv = so3_exp(vec) @ e1
        return tinv

    def predict(self, u, dt): # u is the angular velocity measurement
        # update xhat 
        self.Xhat =  self.Xhat @ so3_exp(u * dt)

        # error dynamics
        A = np.zeros((2,2))
       # B = np.array([[0,1,0],[0,0,1]]) @ R # 2 X 3  #flip??
        self.Rlocal = self.B @ self.Rcov @ self.B.T
        M = self.B @ self.Q @ self.B.T # projected process noise 

        # covariance propagation
        sigma_dot = A @ self.sigma + self.sigma @ A.T + M # 2x2
        self.sigma += sigma_dot * dt
    
    def update(self, g): #g is the gravity measurement 
        # innovation
        e = self.phi(self.Xhat, g)
        innov = self.theta(e) # lical e, epsilon

        # compute state and output gain matrices
        S = self.Cts @ self.sigma @ self.Cts.T + self.Rlocal
        gain = self.sigma @ self.Cts.T @ np.linalg.inv(S) # 2x2

        delta2d = gain @ innov # 2x1

        # lift to SO(3)
        self.Delta = np.array([delta2d[0], delta2d[1], 0])
        self.Xhat = self.Xhat @ so3_exp(self.Delta) 

        # cov update
        I = np.eye(2)
        self.sigma = (I - gain @ self.Cts) @ self.sigma
        
        # reset correction
        self.Delta = np.zeros(3)
        # Xinv = np.linalg.inv(self.X)
        # # convert into local coordinates
        # local_e = self.theta(e)
        # E = np.linalg.inv(self.Xhat) @ self.X # group error element
        # D = -so3_wedge(self.xi0) # group action derivative
        # self.Delta = np.linalg.pinv(D) @ self.theta_inv(-self.xi0)

Get rotation Matrices

In [20]:
import numpy as np
import sympy as sp
from astropy.coordinates import SkyCoord
import astropy.units as units

# create initial rotation based o solved image
# roll is rotation of the camera around its own axis
def initial_rotation(ra, dec, roll):
    # coordinate object for the center of the image
    c = SkyCoord(ra=ra*units.degree,dec = dec*units.degree, frame = 'icrs')

    #ra_rad = np.radians(ra)
    #dec_rad = np.radians(dec)

    ra_rad = c.ra.radian
    dec_rad = c.dec.radian

    z_b_eci = np.array([
        np.cos(dec_rad)* np.cos(ra_rad),
        np.cos(dec_rad)* np.sin(ra_rad),
        np.sin(dec_rad)
    ])

    # z_b_eci = np.array([
    #     np.cos(dec)* np.cos(ra),
    #     np.cos(dec)* np.sin(ra),
    #     np.sin(dec)
    # ])
    # celestial north pole
    n = np.array([0.0,0.0,1.0])

    # body x axis & avoid pole singularities
    if abs(np.dot(n, z_b_eci)) > 0.99:
        x_b_eci = np.array([1.0,0.0,0.0])
    else:
        x_b_eci = np.cross(n, z_b_eci)
        x_b_eci /= np.linalg.norm(x_b_eci)

    y_b_eci = np.cross(z_b_eci, x_b_eci)
    
    # DCM to map camera to sky frame 
    R_cam_to_eci = np.column_stack((x_b_eci, y_b_eci, z_b_eci))
    
    # if camera is rotated, rotate around the local Z-axis
    if roll != 0:
        gamma = np.radians(roll)
        R_roll = np.array([
            [np.cos(gamma),np.sin(gamma), 0.0],
            [-np.sin(gamma),  np.cos(gamma), 0.0],
            [0.0,              0.0,             1.0]
        ])
        R_cam_to_eci = R_cam_to_eci @ R_roll
        
    return R_cam_to_eci

def world_gravity(Xhat, gb, gmst_deg):
    # rotate gravity from body to ECI (stars)
    g_eci = Xhat.T @ gb

    # eart shpin rotation matirx
    theta = np.radians(gmst_deg)
    R_stars_to_ecef = np.array([
        [np.cos(theta),np.sin(theta), 0],
        [-np.sin(theta), np.cos(theta), 0],
        [0, 0, 1]
    ])

    # return gravity in the world frame
    gw = R_stars_to_ecef @ g_eci
    return gw

# now, the gravity estimates are in the world frame
# incorporate the ra, dec coordinates to get a position estimation.

def get_latlon(gw):
    gw = gw / np.linalg.norm(gw)
    
    gx = gw[0] # assuming gw is a tall matrx. 
    gy = gw[1]
    gz = gw[2]

    # horizontal magnitude
    hz = np.sqrt(gx**2 + gy**2)

    lon = np.degrees(np.atan2(gy, gx))
    lat = np.degrees(np.atan2(gz, hz))

    return lat, lon

## CALL THE EQF AND USE IT TO GET LAT/LON

In [28]:
## FIRST GENERATE SYNTHETIC GYROSCOPE/BODY GRAVITY MEASUREMENTS
import numpy as np
from scipy.spatial.transform import Rotation as R

# 1. Simulation Setup
sample_rate = 100  # Hz
duration = 1.0 # seconds
t = np.linspace(0, duration, int(sample_rate * duration))
dt = 1.0 / sample_rate

# Gravity vector in world frame (Z-up)
g_world = np.array([0.0, 0.0, 9.81])

# 2. Define Angular Velocities (rad/s) in Body Frame
omega_x = 0.1 * np.sin(2 * np.pi * 0.2 * t)
omega_y = 0.1 * np.cos(2 * np.pi * 0.1 * t)
omega_z = 0.1 * np.sin(2 * np.pi * 0.05 * t)
gyro_true = np.vstack((omega_x, omega_y, omega_z)).T # shape: N, 3

# 3. Numerical Integration to Compute Orientation Trajectory
# Initialize orientation at t=0
rotations = [R.identity()]

for i in range(1, len(t)):
    # Angular velocity step vector
    w = gyro_true[i - 1]
    # Small rotation over time step dt
    delta_rot = R.from_rotvec(w * dt)
    # Update global orientation: R_new = R_old * delta_R
    new_rot = rotations[-1] * delta_rot
    rotations.append(new_rot)

# Convert list of rotations to a SciPy Rotation object
R_traj = R.from_quat([r.as_quat() for r in rotations])

# 4. Project Gravity into Body Frame (Accelerometer Signal)
# R_traj.inv() transforms world frame vectors to body frame
acc_body = R_traj.inv().apply(g_world) # shape N, 3

# normalize them to get gravity measurements
acc_magnitudes = np.linalg.norm(acc_body, axis=1, keepdims=True)

# Divide measured accelerometer vectors by their magnitude
acc_normalized = acc_body / acc_magnitudes # shape N, 3

# # 5. Add Realistic Sensor Noise and Bias
# gyro_bias = np.array([0.01, -0.02, 0.005])  # rad/s
# acc_bias = np.array([0.05, 0.02, -0.03])    # m/s^2

# gyro_noise = np.random.normal(0, 0.005, gyro_true.shape) # White noise
# acc_noise = np.random.normal(0, 0.02, acc_true.shape)

# gyro_measured = gyro_true + gyro_bias + gyro_noise
# acc_measured = acc_true + acc_bias + acc_noise
###################################################################################################################
# CALL EQF TO CALCULATE COORDINATES 
sx = 1e-4
sy = 1e-4
sz = 1e-4
sg = 1e-4

# define sample CONDITIONS to test ##########################################################
time_unix = 1790269908
ra_s = np.array([6.0, 50.0, 53.9])
dec_s = np.array([360.0-327.0, 13.0, 8.4])

ra = 15 * (ra_s[0] + ra_s[1]/60 + ra_s[2]/3600)
dec = dec_s[0] + dec_s[1]/60 + dec_s[2]/3600

J2000 = 946684800.0
time = time_unix - J2000 # put it into J2000 epoch time 


#gravity_path = r"C:\Users\selen\Documents\puthon\590\CNV\CNV\CNV_csvs\Gravity.csv"
#gyro_path = r"C:\Users\selen\Documents\puthon\590\CNV\CNV\CNV_csvs\Gyroscope.csv"

gr = acc_normalized
gy = gyro_true

# gravitydata = gr.values
# gyrodata = gy.values

gravtimes = t
gyrotimes = t

# TIMESERIES DATASETS
# sort out the gravity data:
gzax = gr[:, 2] # shape 3, N
gyax = gr[:, 1]
gxax = gr[:, 0]
g_measurement = np.vstack((gxax, gyax, gzax)) # gmeas is a long matrix shape 3, N

# gyroscope ANGULAR VLOECITY  data
gyz = gy[:, 0]
gyy = gy[:, 1]
gyx = gy[:, 2]
gyro_data =  np.vstack((gyx, gyy, gyz)) # a long matrix 

steps = g_measurement.shape[1]

X0 = initial_rotation(ra, dec,0)

Xhat_storage = [X0.copy()]
dt = np.abs(gravtimes[1]-gravtimes[0])
dt2 = np.abs(gyrotimes[1] - gyrotimes[0])

eqf = Equivariant(dt2, sigma_x=sx, sigma_y=sy, sigma_z = sz, sigma_gravity = sg, Xinit=X0)

gmst_deg = .004167 * (24110.54841 + 8640184.812866 * time + .093104*(time**2) - (6.2e-6)*(time**3) )
gmst_deg = gmst_deg % 360
gw = np.zeros((steps, 3)) # initialize gw as a tall matrix 

# filtering loop
for k in range(steps):
    g = g_measurement[:, k]
    omega = gyro_data[:, k]

    g = g / np.linalg.norm(g)
    eqf.predict(omega, dt2)
    eqf.update(g)
    
    Xhat_storage.append(eqf.Xhat)
    gmst_deg_k = (gmst_deg + (15.041/3600.0) * (k * dt2)) % 360 # account for earth rotation

    gw[k,:] = world_gravity(eqf.Xhat, g, gmst_deg_k) 

gwmean_x = np.mean(gw[:, 0])
gwmean_y = np.mean(gw[:, 1])
gwmean_z = np.mean(gw[:, 2])
gwmean_final = np.array([gwmean_x, gwmean_y, gwmean_z])  # average the world frame gravity vectors 

gwmean_final = np.mean(gw, axis=0)  # Shape: (3,)

lat, lon = get_latlon(gwmean_final)
# lat /= 2
# lon /= 2

dir_long = "West" if lon > 0 else "East"
dir_lat = "North" if lat > 0 else "South"

lon = np.abs(lon)
lat = np.abs(lat)

print("Latitude: ", lat, dir_lat)
print("Longitude: ", lon, dir_long)



Latitude:  11.522583555154045 North
Longitude:  69.18020532292097 East
